In [ ]:
///- --package .

#### vocations (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:! knight = ()
:! paladin = ()
:! sorcerer = ()
:! druid = ()

:| healer v =
    | PaladinHeals :: healer paladin
    | SorcererHeals :: healer sorcerer
    | DruidHeals :: healer druid

:== heal = cooldowns::healing

:== strike = cooldowns::attack

:| spell v g =
    | WoundCleansing :: spell knight heal
    | Berserk :: spell knight strike
    | DivineHealing :: spell paladin heal
    | LightHealing :: \* v. healer v -> spell v heal
    | IntenseHealing :: \* v. healer v -> spell v heal

:== spell_data =
    {
        words : string
        level : i32
        mana : i32
        cooldown_ms : i32
        group_cooldown_ms : i32
    }

:= healing (words : string) (level : i32) (mana : i32) : spell_data =
    { words = words; level = level; mana = mana; cooldown_ms = 1000; group_cooldown_ms = 1000 }

:= berserk () : spell_data = { words = "exori"; level = 35; mana = 115; cooldown_ms = 4000; group_cooldown_ms = 2000 }

:= data \* v g. (s : spell v g) =
    ?= s ?|
    | WoundCleansing => healing "exura ico" 8 40
    | Berserk => berserk ()
    | DivineHealing => healing "exura san" 35 160
    | LightHealing _ => healing "exura" 8 20
    | IntenseHealing _ => healing "exura gran" 20 70

:= cooldown_id_of \* v g. (s : spell v g) : cooldowns::spell_id g =
    :< cooldowns
    ?= s ?|
    | WoundCleansing => WoundCleansingSpell
    | Berserk => StrikeSpell
    | DivineHealing | LightHealing _ | IntenseHealing _ => LightHealingSpell

:= as_cooldown \* v g. (s : spell v g) : cooldowns::spell g =
    := d = data s
    { id = cooldown_id_of s; own_ms = d::cooldown_ms; group_ms = d::group_cooldown_ms }

:= castable \* v g. (level : i32) (mana : i32) (s : spell v g) : bool =
    := d = data s
    level >= d::level && mana >= d::mana

:= knight_heal () : spell knight heal = WoundCleansing

#### spell_levels_and_mana_gate_casting_like_canary (test)

In [1]:
///- --test --symbol
///> c

:= ~level = 8i32
castable level 90 WoundCleansing |> _assert_eq !+
castable level 90 Berserk |> _assert_eq !-
castable (level + 12) 70 (IntenseHealing DruidHeals) |> _assert_eq !+

__assert_eq { expected = true }
__assert_eq { expected = false }
__assert_eq { expected = true }

#### spell_data_matches_canary_scripts (test)

In [2]:
///- --test --symbol
///> c

:= ~healer = PaladinHeals
(data (LightHealing healer))::mana |> _assert_eq 20i32
(data Berserk)::cooldown_ms |> _assert_eq 4000i32

__assert_eq { expected = 20 }
__assert_eq { expected = 4000 }

#### a_knight_cannot_cast_light_healing (test)

In [3]:
///- --test --symbol --fails "Unification failure"
///> c

:= s : spell knight heal = LightHealing PaladinHeals
(data s)::mana

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/tibia/tibia/src/vocations.spi:80:45 (main.spi:64:45): ETY-TermError: Unification failure. Got:      healer knight Expected: healer paladin In the definition: spiral_cells_test.


#### a_paladin_cannot_cast_wound_cleansing (test)

In [4]:
///- --test --symbol --fails "Unification failure"
///> c

:= s : spell paladin heal = WoundCleansing
(data s)::mana

failed to compile as expected ("Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/tibia/tibia/src/vocations.spi:85:33 (main.spi:64:33): ETY-TermError: Unification failure. Got:      spell paladin healing Expected: spell knight healing In the definition: spiral_cells_test.


#### spell_data_on_gpu (test)

In [5]:
///- --test --symbol
///> wgsl

:= ~level = 8i32
? castable level 90 WoundCleansing && castable level 90 Berserk = !- && (data Berserk)::cooldown_ms = 4000i32
?> 0i32
|: 1i32